In [1]:
import os
from langchain_community.document_loaders import TextLoader
# We go back to the most stable import for Chroma
from langchain_community.vectorstores import Chroma 
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

# File and Model info
FILE_PATH = "data.txt"
MODEL_NAME = "llama3"
DB_DIR = "./chroma_db"

# Create the data
sample_text = """
VACATION DESTINATION: Blue Lagoon Island
ACCOMMODATION: The "Sunset Hut" is the best room. It has a blue door and a hammock.
ACTIVITIES: Morning: Scuba diving. Afternoon: Coconut bowling. Evening: Fire dancing.
FOOD: The signature drink is the "Pineapple Punch" which costs $5. 
The local specialty is Spicy Mango Shrimp. Avoid the "Seaweed Soup."
VIBE: The resident cat is named "Captain Whiskers."
"""
with open(FILE_PATH, "w") as f:
    f.write(sample_text)

print("✅ Step 1: System reset. Everything is ready.")

/Users/keerthanaaalagudurai/AI_Projects/my-rag-project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Step 1: System reset. Everything is ready.


In [2]:
loader = TextLoader(FILE_PATH)
docs = loader.load()

# 500 characters ensures "Sunset Hut" and "best room" stay together
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(f"✅ Step 2: Created {len(chunks)} chunks.")

✅ Step 2: Created 1 chunks.


In [3]:
import os
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain.chains import RetrievalQA

# 1. Configuration Check
MODEL_NAME = "llama3"
EMBEDDING_MODEL = "nomic-embed-text" # Specialized embedding model

# 2. Init AI Models
print("Warming up the M3 GPU...")
# Now using the specialized model for embeddings
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL) 
llm = OllamaLLM(model=MODEL_NAME, temperature=0)

# 3. Build Vector Store IN MEMORY 
try:
    print(f"Processing {len(chunks)} chunks using {EMBEDDING_MODEL}...")
    vectorstore = InMemoryVectorStore.from_documents(
        documents=chunks, 
        embedding=embeddings
    )
    
    # 4. Create the search chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 2})
    )
    print("✅ Step 3: RAG Engine is officially LIVE with new embeddings!")

except NameError:
    print("❌ Error: 'chunks' not found. Please run Cell 2 again!")
except Exception as e:
    print(f"❌ Connection Error: {e}")
    print(f"👉 Ensure you have run: ollama pull {EMBEDDING_MODEL}")

Warming up the M3 GPU...
Processing 1 chunks using nomic-embed-text...
✅ Step 3: RAG Engine is officially LIVE with new embeddings!


In [4]:
# Run this cell to ask a question!
query = input("Ask a question about your data: ")

if query:
    print(f"\nSearching for: {query}...")
    response = qa_chain.invoke(query)
    
    print("\n--- AI RESPONSE ---")
    print(response['result'])


Searching for: what is the afternoon plan...

--- AI RESPONSE ---
According to the context, the afternoon plan is Coconut bowling!
